## At-Most-Once
This means that message is delivered zero or once. Message may get lost but never duplicated.  

From the perspective of producer, it means that the producer fires and forgets (doesn't wait for acknowledgment (`acks=0`)).  
From the perspective of consumer, this means that offset is committed before the message is processed. The consumer can crash after committing offset resulting in message being lost.

This setup can be used when occasional message loss is acceptable. Example: logs, telementry data, etc.

## At-Least-Once
In this scenario message is delivered one or more time. Messages are not lost, but can be duplicated.

From the perspective of producer, it means that the producer retries on failure. For this, setting `acks=all` is recommended.  
From the perspective of consumer, this means that offset is committed after the message is processed. The consumer can crash after processing the message, but before commiting it - this can lead to message being processed twice.

This setup is suitable when processing duplicate message is idempotent (upsert write to database).

## Exactly-Once
As the name suggests, this indicates message delivered exactly one time with no loss or duplicates. Kafka achieves this using:
- idempotent producer
- transactions

### Idempotent Producer
Kafka producers are configured to retry if it doesn't receive acknowledgement from broker. Let's say the leader broker received message and then it replicates it to follower brokers. But before responding back to the producer, the leader broker crashes. In this case the producer would resend the same message which will be received by a new leader broker which already had the message - leading to duplicates.

Kafka idempotent producers work by combining:
- **Producer Id:** broker assigns a new id to each unique producer
- **Sequence Number:** each message sent by producer gets assigned an increasing number

When a broker sees that a message with same producer id and sequence number is being stored, it silently rejects the message. To mark a producer as idempotent, add the below configuration to the producer:

In [ ]:
props.put("enable.idempotence", "true");

/*
When enabled, Kafka automatically sets:

acks=all (wait for all in-sync replicas to acknowledge)
retries=Integer.MAX_VALUE (unlimited retries by default)
max.in.flight.requests.per.connection=5 (or less, to maintain ordering)

*/

What happens when the broker receives out of order message? As an example, broker received message in a partition having sequence numbers 1, 2, 3. The next sequence number it received is 5. In such case the broker would reject the message with exception `OutOfOrderSequenceException`.

In [ ]:
try {
    producer.send(new ProducerRecord<>("my-topic", "key", "value"))
        .get(); // Blocks until response
} catch (ExecutionException e) {
    if (e.getCause() instanceof OutOfOrderSequenceException) {
        // This is a fatal error for idempotent producer
        // The producer must be closed and recreated
        System.err.println("FATAL: Out of sequence detected!");
        producer.close();
        // Create new producer or handle gracefully
    }
}

### Transactions
Transactions in Kafka were developed specifically for stream processing applications, and works in context of *consume-process-produce* pattern where you consume message from a topic, process it and then publish it to another topic. Let's see a few scenarios:

1. Message was consumed, processed and published to output queue. But the application crashed before committing the offset.
2. Message was consumed, application got stuck processing it. Another application instance replaced the stuck instance, reprocessed the message and published the message to output queue. Now, the stuck instance recovered and published duplicate message to the same queue.

Kafka transactions introduce the concept of *atomic multi-partition write*. Either all messages written across multiple partitions/topics become visible together, or none do.

To use Kafka transactions, we use *transactional producer*. To create such producers:

In [ ]:
Properties props = new Properties();
props.put("bootstrap.servers", "localhost:9092");
props.put("transactional.id", "my-app-producer-1");
props.put("enable.idempotence", "true"); // Automatically enabled

KafkaProducer<String, String> producer = new KafkaProducer<>(props);
producer.initTransactions(); // Critical: Initialize on startup